## CHATBOT WITH CONVERSATION HISTORY

Project Name -- ChatbotWithConversationHistory

- In this vides, we will go over an example of how to design and implement a LLM powered chatbot. This chatbot will be able to have a conversation and remeber previous interactions.

- Note, that this chatbot that we will build only use the language model to have a conversation. There are several other related topics that you might be looking for:- 

1. Conversation RAG: enable a chatbot experience over an external source of data
2. Agent: build a chatbot that can take actions

This will only cover the basics which will be helpful for the above two advanced topics

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

## For Groq API
groq_api_key = os.getenv("GROQ_API_KEY")

## For LangSmith Tracking
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY") or os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGSMITH_PROJECT"] = os.getenv("LANGSMITH_PROJECT") or os.getenv("LANGCHAIN_PROJECT")
 
# Usually optional for hosted LangSmith, but safe to set
os.environ["LANGSMITH_ENDPOINT"] = os.getenv("LANGSMITH_ENDPOINT")

In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key)
model

/Users/anushkjain/Desktop/CODING/GEN_AI/07_langchain/venv/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/Users/anushkjain/Desktop/CODING/GEN_AI/07_langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x16cb41210>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x16cb43640>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi, My name is Anushk and I am a Chief AI Engineer.")])

AIMessage(content="Nice to meet you, Anushk. As a Chief AI Engineer, I'm sure you're at the forefront of developing and implementing cutting-edge AI solutions. What brings you here today? Are you looking for assistance with a specific project or perhaps seeking information on the latest advancements in the field of AI?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 51, 'total_tokens': 113, 'completion_time': 0.086137625, 'completion_tokens_details': None, 'prompt_time': 0.002952256, 'prompt_tokens_details': None, 'queue_time': 0.161090651, 'total_time': 0.089089881}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ed0f0-ff8c-7961-aa8f-ded89b3ee1e1-0', usage_metadata={'input_tokens': 51, 'output_tokens': 62, 'total_tokens': 113})

In [5]:
from langchain_core.messages import AIMessage

## Whatever message I was giving here, the model is able to remember the context
model.invoke([
    HumanMessage(content="Hi, My name is Anushk and I am a Chief AI Engineer."),
    AIMessage(content="Nice to meet you, Anushk. As a Chief AI Engineer, you must be at the forefront of innovation in the field of artificial intelligence. What specific aspects of AI are you currently working on or interested in?"),
    HumanMessage(content="Hey, What is my name? and what do I do?")
])

AIMessage(content='Your name is Anushk, and you are a Chief AI Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 118, 'total_tokens': 134, 'completion_time': 0.017126793, 'completion_tokens_details': None, 'prompt_time': 0.015903449, 'prompt_tokens_details': None, 'queue_time': 0.05110063, 'total_time': 0.033030242}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ed0f1-01aa-7151-aece-65dae7aaa2c4-0', usage_metadata={'input_tokens': 118, 'output_tokens': 16, 'total_tokens': 134})

### Message History

Message history is the sequence of previous user, assistant, and system messages that are included in an LLM request to provide conversational context. The LLM itself does not retain memory between calls; the application stores the history and resends relevant messages with each new request so the model can generate context-aware responses.

We can use the MessageHistory Class to wrap our model and make it stateful. This will keep the track of the inputs and outputs of the model, and store them in some datastores. Futher interactions will then load these messages and pass them into the chain as part of the input.

In [6]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

## now as we use chatgpt, it is used to maintain different sessions for different user that are chatting separately, we can do this by creating this function 
## here, the name of the function is get_session_history, session_id is the parameter which is of str type and returns a BaseChatMessageHistory object
## BaseChatMessageHistory --> abstract class for storing messaged history
## ChatMessageHistory --> in memory implementation of chat message history. Store messages in a memory list.

store = {
    
}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    
    ## if session_id is not present in the store, then we will create a session_id as a key and it will store the value of ChatMessageHistory object as value. This value will store all the messages.
    ## Also ChatMessageHistory IS A BaseChatMessageHistory
    
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)
    

/Users/anushkjain/Desktop/CODING/GEN_AI/07_langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3577: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [7]:
## for now let me hard code the session_id

config = {"configurable": {"session_id": "chat1"}}

In [8]:
with_message_history.invoke([
    HumanMessage(content="Hi, My name is Anushk and I am a Chief AI Engineer.")
],  config=config)

AIMessage(content="Nice to meet you, Anushk. As a Chief AI Engineer, I'm sure you have a fascinating role that involves designing, developing, and implementing artificial intelligence and machine learning solutions. What specific areas of AI are you currently working on, and what exciting projects are you undertaking?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 51, 'total_tokens': 109, 'completion_time': 0.102608907, 'completion_tokens_details': None, 'prompt_time': 0.003033254, 'prompt_tokens_details': None, 'queue_time': 0.158833803, 'total_time': 0.105642161}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ed0f1-026b-75e1-863b-2229d199be5c-0', usage_metadata={'input_tokens': 51, 'output_tokens': 58, 'total_tokens': 109})

In [9]:
with_message_history.invoke([HumanMessage(content="What is my name")], config=config)

AIMessage(content='Your name is Anushk.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 122, 'total_tokens': 130, 'completion_time': 0.01338244, 'completion_tokens_details': None, 'prompt_time': 1.029766429, 'prompt_tokens_details': None, 'queue_time': 0.451600262, 'total_time': 1.043148869}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ed0f1-0432-7e51-be41-d06aa8d7786b-0', usage_metadata={'input_tokens': 122, 'output_tokens': 8, 'total_tokens': 130})

### Prompt Templates

- Prompt templates help to turn raw user information into the format that LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. let's now make it a little more complicated. First, add in a custom message with some custom instructions (but still taking messages as a input). Next, we'll add in more inputs beside just the messages.

In [10]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer all the question to the best of your knowledge"),
    MessagesPlaceholder(variable_name="messages")
])

chain = prompt | model

In [11]:
## we will use the message placeholder to hold the list of human messages.
chain.invoke({"messages": [HumanMessage(content="Hi, My name is Anushk")]})

AIMessage(content="Nice to meet you, Anushk. I'm happy to assist you with any questions or topics you'd like to discuss. Is there something specific on your mind, or would you like to start with a conversation?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 59, 'total_tokens': 104, 'completion_time': 0.08126516, 'completion_tokens_details': None, 'prompt_time': 0.003530144, 'prompt_tokens_details': None, 'queue_time': 0.200423512, 'total_time': 0.084795304}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ed0f1-0a8c-79c2-b129-ff645d53ef92-0', usage_metadata={'input_tokens': 59, 'output_tokens': 45, 'total_tokens': 104})

In [12]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

/Users/anushkjain/Desktop/CODING/GEN_AI/07_langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3577: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [13]:
config={"configurable": {"session_id": "chat3"}}

## Here langchain doing the inside work, it places the current input along with all the message history in the messages variable that we created in the prompt template
response = with_message_history.invoke([HumanMessage(content="Hi, My name is Anushk")], config=config)

In [14]:
response.content

'Nice to meet you, Anushk. How can I assist you today?'

In [15]:
response = with_message_history.invoke([HumanMessage(content="what is my name?")], config=config)
response.content

'Your name is Anushk.'

In [16]:
## adding more complexity in the prompt, here we are adding another variable language in the prompt

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer all the question to the best of your knowledge in {language}"),
    MessagesPlaceholder(variable_name="messages")
])

chain = prompt | model

In [17]:
## now as we have multiple variables in the prompt we have invoke the chain by putting each variable's value
response = chain.invoke({"messages": [HumanMessage(content="can you tell me my name again?")], "language": "Hindi"})
response

AIMessage(content='मुझे खेद है, लेकिन मैं आपके नाम को याद नहीं कर सकता। हमारी पहली बातचीत हो रही है, इसलिए मेरे पास आपके बारे में कोई जानकारी नहीं है। कृपया अपना नाम बताएं और मैं आपकी मदद करने के लिए तैयार हूं।', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 61, 'total_tokens': 156, 'completion_time': 0.12326845, 'completion_tokens_details': None, 'prompt_time': 0.003538344, 'prompt_tokens_details': None, 'queue_time': 0.158842323, 'total_time': 0.126806794}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019ed0f1-0d9f-71b1-915c-f8546c59f0e5-0', usage_metadata={'input_tokens': 61, 'output_tokens': 95, 'total_tokens': 156})

Let's wrap this chain with the MessageHistory class in order to send the chat history as well. As there are multiple keys in the input, we need to correctly specify the correct key to use to save the chat history

In [18]:
with_message_history = RunnableWithMessageHistory(chain,
                                                  get_session_history,
                                                  input_messages_key="messages")

/Users/anushkjain/Desktop/CODING/GEN_AI/07_langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3577: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [19]:
response = with_message_history.invoke({"messages": [HumanMessage(content="can you tell me my name again?")], "language": "Hindi"}, config=config)
response.content

'अपना नाम है - अनुष्क'

In [20]:
response.content

'अपना नाम है - अनुष्क'

### Managing the Conversation History

One important concept to understand when building chatbot is managing the conversation history. If left unmanaged, the list of messages will grow unbound and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of messages you are passing in.

#### Why Not Just Send Everything?

Three reasons:

1. Context Window Limits

Every model has a maximum context size.

Examples:

Llama 3 → depends on version
GPT-4o → large context window
Gemma → smaller context window

You cannot exceed it

2. Cost

If using APIs:

100 messages

costs much more than:

10 messages

because every request resends the history.

3. Speed

More tokens:

 Longer Prompt
    ↓
 More Processing
    ↓
 Slower Response

- trim_messages helps to reduce how many messages we are sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with the other parameters like if we always want to keep the system messages, or whether we allow the partial messages.

In [24]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=70,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(
        content="You are a helpful AI assistant."
    ),

    HumanMessage(
        content="Hi, my name is Anushk."
    ),

    AIMessage(
        content="Hello Anushk! Nice to meet you."
    ),

    HumanMessage(
        content="I am learning LangChain."
    ),

    AIMessage(
        content="That's great! LangChain is a framework for building LLM applications."
    ),

    HumanMessage(
        content="What is a Retriever?"
    ),

    AIMessage(
        content="A Retriever fetches relevant documents from a knowledge source based on a query."
    )
]

trimmer.invoke(messages)


[SystemMessage(content='You are a helpful AI assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I am learning LangChain.', additional_kwargs={}, response_metadata={}),
 AIMessage(content="That's great! LangChain is a framework for building LLM applications.", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is a Retriever?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='A Retriever fetches relevant documents from a knowledge source based on a query.', additional_kwargs={}, response_metadata={})]

In [27]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer) | prompt | model
)

response = chain.invoke({"messages": messages + [HumanMessage(content="What's my name?")], "language": "Hindi"})
response.content

'मेरे पास आपके नाम के बारे में जानकारी नहीं है। आप मुझे बता सकते हैं कि आपका नाम क्या है?'

In [23]:
## Let's wrap it in the message history
with_message_history = RunnableWithMessageHistory(chain,
                                                  get_session_history,
                                                  input_messages_key="messages")

config = {"configurable": {"session_id": "chat5"}}

/Users/anushkjain/Desktop/CODING/GEN_AI/07_langchain/venv/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3577: LangChainPendingDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [26]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="What's my name?")],
        "language": "Hindi"
    },
    config=config
)

response.content

'आपका नाम मुझे नहीं पता क्योंकि हमारी बातचीत शुरू हुई है। आप LangChain के बारे में जानने के लिए यहाँ हैं।'